# RAG for Structured Data (CSV) — From First Principles

## Overview

This notebook is a deep, hands-on exploration of **Retrieval-Augmented Generation (RAG)** applied specifically to **structured and tabular data** stored in CSV format. While most RAG tutorials focus on unstructured text (PDFs, articles, web pages), real-world enterprise data overwhelmingly lives in structured formats — databases, spreadsheets, and CSV exports. Applying RAG to this kind of data requires fundamentally different strategies than the "chunk-and-embed" approach used for prose.

We will build a complete CSV RAG pipeline from scratch using **no frameworks** (no LangChain, no LlamaIndex) — just native Python with `pandas`, `sentence-transformers`, `FAISS`, and a locally-quantized LLM (`Qwen3-14B`). By the end, you will understand:

1. **Why structured data is fundamentally different** from unstructured text for RAG
2. **Multiple serialization strategies** for converting table rows into embeddable text
3. **How to build a FAISS-backed retrieval pipeline** optimized for tabular data
4. **Advanced techniques** like hybrid filtering, column-specific queries, and numerical handling
5. **When CSV RAG is the right tool** and when you should use SQL instead

## Why Structured Data Is Different from Unstructured Text

When we apply RAG to a PDF or a Wikipedia article, we're working with **flowing natural language** — paragraphs, sentences, narrative structure. The text is self-contained: a paragraph about climate change *reads* like text about climate change. Embedding models, which are trained on natural language, handle this effortlessly.

**Structured data breaks all of these assumptions.** A CSV row like `TechCorp,450,5000,Technology` is meaningless without its schema. The number `450` could be revenue, employees, or a zip code — the column header gives it meaning. Embedding models trained on natural language will struggle with raw tabular data because:

- **Column semantics are implicit**: The value "450" means nothing without knowing it represents revenue in millions
- **Relationships are positional**: In a table, meaning comes from the column a value occupies, not from surrounding words
- **Data types are mixed**: Numbers, categories, dates, and free text coexist in the same row
- **Redundancy is low**: Unlike prose where an idea is often restated multiple ways, each cell contains exactly one fact

This means our core challenge is **representation** — how do we convert a structured row into text that an embedding model can meaningfully encode? The answer to this question determines the quality of our entire RAG pipeline.

## Approaches to Tabular RAG

There are several strategies for making structured data retrievable via semantic search, each with different tradeoffs:

| Approach | Description | Pros | Cons |
|----------|-------------|------|------|
| **Row-as-document** | Serialize each row into a natural language sentence | Simple, preserves row-level semantics | Loses inter-row relationships |
| **Column-aware chunking** | Prepend column names to values in structured format | Preserves schema context | Can feel mechanical to embedding models |
| **Schema-enriched embeddings** | Add metadata descriptions to column values | Best embedding quality | Requires manual schema annotation |
| **Group-level chunking** | Aggregate rows by category before embedding | Captures group patterns | Loses individual row details |
| **Hybrid (filter + search)** | Use structured filters first, then semantic search | Best of both worlds | More complex implementation |

In this notebook, we will implement and compare all five approaches. We will see that **the serialization strategy matters enormously** — the same data embedded with different text representations can yield dramatically different retrieval quality.

## When to Use RAG over SQL

A natural question arises: if the data is structured, why not just use SQL? The answer is nuanced:

**Use RAG when:**
- Users ask questions in natural language ("Tell me about companies in healthcare")
- Queries require semantic understanding ("Which companies are similar to TechCorp?")
- The data contains free-text fields (descriptions, notes, comments)
- You want to combine retrieval with LLM-powered synthesis and reasoning
- Users don't know the schema and can't write structured queries

**Use SQL when:**
- Queries are purely numerical ("What is the average revenue?", "Count employees > 1000")
- Exact matching is required ("Show me row where company = 'TechCorp'")
- Aggregations are the primary need (SUM, AVG, GROUP BY)
- The query can be precisely translated to SQL

**The sweet spot for CSV RAG** is when you have mixed data — some structured columns (numbers, categories) and some unstructured columns (descriptions, notes) — and users want to ask natural language questions that blend semantic understanding with factual retrieval. This is exactly what we'll build.

---
# Setup: Environment & Models

The following four cells install dependencies, load our quantized LLM, load the embedding model, and define a generation helper. These are standard boilerplate for our Colab-based RAG notebooks.

In [ ]:
# Cell 1: Install all dependencies
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy pandas

In [ ]:
# Cell 2: Mount Google Drive & Load Quantized LLM
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto"
)
print(f"\u2705 Loaded {MODEL_NAME} with 4-bit NF4 quantization")

In [ ]:
# Cell 3: Load Embedding Model
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)
print(f"\u2705 Embedding model loaded \u2014 dimension: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
# Cell 4: Generation Helper
def generate(prompt, max_new_tokens=512):
    """Send a prompt to Qwen3-14B and return the generated text."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        top_k=20,
        temperature=0.7
    )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

# Quick smoke test
print(generate("Say 'Hello, CSV RAG!' in one sentence.", max_new_tokens=32))

---
# Section 2: CSV Parsing & Representation

## Creating Our Synthetic Dataset

For this notebook, we'll work with a synthetic dataset of companies. Using inline synthetic data (rather than downloading external files) ensures full reproducibility and lets us carefully control the data characteristics. Our dataset mixes **categorical columns** (sector, founded year), **numerical columns** (revenue, employees), and **free-text columns** (description) — exactly the kind of mixed-type data where CSV RAG shines.

In [ ]:
import pandas as pd
import numpy as np

data = {
    'company': [
        'TechCorp', 'GreenEnergy', 'HealthPlus', 'FinanceHub',
        'EduLearn', 'AutoDrive', 'CloudNine', 'BioGenesis',
        'RetailMax', 'SpaceVentures', 'AgriTech', 'MediaFlow',
        'CyberShield', 'QuantumLabs', 'FoodChain'
    ],
    'revenue_millions': [
        450, 120, 89, 780, 45, 340, 560, 210, 920, 75, 63, 190, 310, 28, 410
    ],
    'employees': [
        5000, 800, 1200, 3500, 300, 4200, 2800, 950, 8500, 150, 420, 1100, 1800, 85, 6200
    ],
    'sector': [
        'Technology', 'Energy', 'Healthcare', 'Finance',
        'Education', 'Automotive', 'Technology', 'Healthcare',
        'Retail', 'Aerospace', 'Agriculture', 'Media',
        'Cybersecurity', 'Technology', 'Food & Beverage'
    ],
    'founded': [
        2005, 2012, 2008, 1998, 2015, 2010, 2014, 2011, 1995, 2018, 2009, 2013, 2016, 2020, 2001
    ],
    'hq_city': [
        'San Francisco', 'Austin', 'Boston', 'New York',
        'Chicago', 'Detroit', 'Seattle', 'San Diego',
        'Dallas', 'Houston', 'Des Moines', 'Los Angeles',
        'Washington DC', 'Boulder', 'Atlanta'
    ],
    'description': [
        'Leading AI software company specializing in enterprise machine learning platforms and natural language processing solutions for Fortune 500 clients.',
        'Renewable energy startup focused on next-generation solar panel technology and grid-scale battery storage for residential and commercial use.',
        'Digital health platform providing telemedicine services, AI-powered diagnostics, and remote patient monitoring for rural and underserved communities.',
        'Established financial services firm offering algorithmic trading platforms, risk analytics, and wealth management tools for institutional investors.',
        'EdTech startup building adaptive learning platforms that use AI to personalize curriculum for K-12 students across multiple subjects.',
        'Autonomous vehicle company developing self-driving technology for commercial trucks and last-mile delivery vehicles in urban environments.',
        'Cloud infrastructure provider offering serverless computing, container orchestration, and multi-cloud management tools for enterprise DevOps teams.',
        'Biotechnology firm focused on CRISPR gene editing therapies for rare genetic disorders and personalized cancer treatments.',
        'Largest omnichannel retail company in the Southwest, operating 500+ stores with advanced supply chain optimization and demand forecasting.',
        'Commercial space startup building small satellite launch vehicles and orbital debris cleanup technology for low Earth orbit missions.',
        'Precision agriculture company using drone imagery, IoT soil sensors, and machine learning to optimize crop yields and reduce water usage.',
        'Digital media conglomerate operating streaming platforms, podcast networks, and AI-powered content recommendation engines for 50 million subscribers.',
        'Enterprise cybersecurity firm providing zero-trust network architecture, threat intelligence, and automated incident response for government agencies.',
        'Quantum computing research lab developing fault-tolerant qubits and quantum algorithms for drug discovery and materials science applications.',
        'Farm-to-table food distribution company using blockchain traceability and cold chain IoT monitoring to deliver fresh produce across 30 states.'
    ]
}

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} companies, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
print(f"Sectors: {df['sector'].nunique()} unique")
print()
df

## Row-to-Text Conversion: The Heart of Tabular RAG

The single most important decision in tabular RAG is **how you convert each row into text** for embedding. This is called **serialization** — turning structured data into a string that an embedding model can process. Let's explore and compare several strategies.

**Why does this matter so much?** Embedding models like BGE are trained on natural language text — sentences, paragraphs, documents. When we feed them `"TechCorp,450,5000,Technology"`, they have no training signal to understand what those values mean. But if we feed them `"TechCorp is a Technology company with $450M revenue and 5,000 employees"`, the model can leverage its understanding of English to create a meaningful vector.

The quality of your serialization directly determines:
1. **Retrieval precision** — will the right rows be found for a query?
2. **Semantic clustering** — will similar companies be near each other in vector space?
3. **Query compatibility** — will user queries be close to relevant row embeddings?

In [ ]:
# Strategy 1: Naive CSV-style (baseline \u2014 what most beginners do)
def serialize_naive(row):
    """Just join values with commas \u2014 loses all schema context."""
    return ', '.join(str(v) for v in row.values)

# Strategy 2: Key-value pairs (preserves column names)
def serialize_key_value(row):
    """Structured key: value format \u2014 preserves schema but feels mechanical."""
    return ' | '.join(f"{col}: {val}" for col, val in row.items())

# Strategy 3: Natural language sentence (best for embedding models)
def serialize_natural_language(row):
    """Convert row to a natural English sentence \u2014 matches embedding model training."""
    return (
        f"{row['company']} is a {row['sector']} company headquartered in {row['hq_city']}, "
        f"founded in {row['founded']}. It has {row['employees']:,} employees and generates "
        f"${row['revenue_millions']}M in annual revenue. {row['description']}"
    )

# Strategy 4: Schema-enriched (adds semantic context to column values)
def serialize_schema_enriched(row):
    """Add explicit semantic labels for maximum clarity."""
    return (
        f"Company Name: {row['company']}\n"
        f"Industry Sector: {row['sector']}\n"
        f"Headquarters Location: {row['hq_city']}\n"
        f"Year Founded: {row['founded']}\n"
        f"Number of Employees: {row['employees']:,}\n"
        f"Annual Revenue (USD Millions): ${row['revenue_millions']}M\n"
        f"Business Description: {row['description']}"
    )

# Show all four strategies for the first row
sample_row = df.iloc[0]
strategies = {
    'Naive CSV': serialize_naive,
    'Key-Value Pairs': serialize_key_value,
    'Natural Language': serialize_natural_language,
    'Schema-Enriched': serialize_schema_enriched
}

for name, func in strategies.items():
    text = func(sample_row)
    print(f"\n{'='*60}")
    print(f"\U0001f4cb Strategy: {name}")
    print(f"   Length: {len(text)} chars")
    print(f"{'='*60}")
    print(text)

## Column Metadata Injection

Beyond row-level serialization, we can inject **global schema metadata** into each chunk. This technique tells the embedding model not just *what* the values are, but *what kind of data* the table contains. Think of it as giving the model a "table of contents" before showing it each row.

This is especially powerful when column names are abbreviated or ambiguous. For example, a column named `rev_m` could mean "revenue in millions" or "revisions per month" — injecting a metadata header disambiguates this for the embedding model. Even when column names are clear, the metadata header helps the embedding model create vectors that are more aligned with the kinds of questions users will ask.

In [ ]:
# Create a schema description from the DataFrame
def build_schema_description(dataframe):
    """Generate a human-readable schema summary for metadata injection."""
    lines = ["This data contains information about companies with the following attributes:"]
    for col in dataframe.columns:
        dtype = dataframe[col].dtype
        if dtype in ['int64', 'float64']:
            min_val, max_val = dataframe[col].min(), dataframe[col].max()
            lines.append(f"  - {col} ({dtype}): ranges from {min_val} to {max_val}")
        else:
            n_unique = dataframe[col].nunique()
            sample = dataframe[col].unique()[:3].tolist()
            lines.append(f"  - {col} ({dtype}): {n_unique} unique values, e.g., {sample}")
    return '\n'.join(lines)

schema_desc = build_schema_description(df)
print(schema_desc)

# Strategy 5: Schema-metadata + natural language (the premium approach)
def serialize_with_metadata(row, schema):
    """Prepend the schema description to each row\'s natural language form."""
    nl_text = serialize_natural_language(row)
    return f"[TABLE CONTEXT]\n{schema}\n\n[ROW DATA]\n{nl_text}"

# Show the metadata-enriched version
print("\n" + "="*60)
print("\U0001f4cb Strategy: Schema-Metadata + Natural Language")
print("="*60)
print(serialize_with_metadata(df.iloc[0], schema_desc))

---
# Section 3: Chunking Strategies for Tabular Data

In traditional RAG for documents, "chunking" means splitting long text into manageable pieces. For tabular data, the concept is different — each row is already a natural unit. But the question of *granularity* still matters: should each row be its own chunk? Should we group rows by category? Should we include multiple rows per chunk to preserve inter-row context?

Each strategy creates different retrieval behaviors:
- **Row-level chunks** are precise but can miss patterns that span multiple rows
- **Group-level chunks** capture category-level patterns but are less precise for individual entity queries
- **Schema-enriched chunks** are more expensive to embed but produce higher-quality vectors

Let's implement all three and build the infrastructure to compare them.

In [ ]:
# Chunking Strategy 1: Row-level (each row = one chunk)
def chunk_by_row(dataframe, serializer):
    """Create one chunk per row using the specified serialization function."""
    chunks = []
    for idx, row in dataframe.iterrows():
        chunks.append({
            'text': serializer(row),
            'metadata': {'row_index': idx, 'company': row['company'], 'sector': row['sector']},
            'strategy': 'row-level'
        })
    return chunks

# Chunking Strategy 2: Group-level (aggregate by sector)
def chunk_by_group(dataframe, group_col='sector'):
    """Create one chunk per group, summarizing all rows in that group."""
    chunks = []
    for group_name, group_df in dataframe.groupby(group_col):
        companies = group_df['company'].tolist()
        avg_rev = group_df['revenue_millions'].mean()
        total_emp = group_df['employees'].sum()
        descriptions = ' '.join(group_df['description'].tolist())
        text = (
            f"The {group_name} sector contains {len(companies)} companies: "
            f"{', '.join(companies)}. The average revenue is ${avg_rev:.0f}M "
            f"with {total_emp:,} total employees. "
            f"Business activities include: {descriptions}"
        )
        chunks.append({
            'text': text,
            'metadata': {'sector': group_name, 'companies': companies, 'count': len(companies)},
            'strategy': 'group-level'
        })
    return chunks

# Chunking Strategy 3: Schema-enriched row-level
def chunk_schema_enriched(dataframe, serializer, schema):
    """Row-level chunks with schema metadata prepended."""
    chunks = []
    for idx, row in dataframe.iterrows():
        text = f"[TABLE SCHEMA]\n{schema}\n\n[ROW]\n{serializer(row)}"
        chunks.append({
            'text': text,
            'metadata': {'row_index': idx, 'company': row['company'], 'sector': row['sector']},
            'strategy': 'schema-enriched'
        })
    return chunks

# Generate all three chunk sets
row_chunks = chunk_by_row(df, serialize_natural_language)
group_chunks = chunk_by_group(df)
schema_chunks = chunk_schema_enriched(df, serialize_natural_language, schema_desc)

print(f"Row-level chunks:      {len(row_chunks)} (1 per company)")
print(f"Group-level chunks:    {len(group_chunks)} (1 per sector)")
print(f"Schema-enriched chunks: {len(schema_chunks)} (1 per company + schema)")
print(f"\nSample row chunk length:      {len(row_chunks[0]['text'])} chars")
print(f"Sample group chunk length:    {len(group_chunks[0]['text'])} chars")
print(f"Sample schema chunk length:   {len(schema_chunks[0]['text'])} chars")

## Comparing Embedding Quality Across Serialization Strategies

Let's quantitatively measure how different serialization strategies affect embedding quality. We'll embed the same data using four strategies and check whether **semantically similar companies** (same sector) end up closer together in vector space than **dissimilar companies** (different sectors).

A good serialization strategy should produce embeddings where intra-sector cosine similarity is higher than inter-sector similarity — meaning the embedding model "understands" that companies in the same sector are related.

In [ ]:
import numpy as np

# Embed the same rows using different serialization strategies
serializers = {
    'Naive': serialize_naive,
    'Key-Value': serialize_key_value,
    'Natural Language': serialize_natural_language,
    'Schema-Enriched': serialize_schema_enriched
}

def compute_sector_coherence(dataframe, serializer, emb_model):
    """Measure how well embeddings cluster by sector.

    Returns the gap between average intra-sector similarity and
    average inter-sector similarity (higher = better clustering).
    """
    texts = [serializer(row) for _, row in dataframe.iterrows()]
    embeddings = emb_model.encode(texts, normalize_embeddings=True)
    sectors = dataframe['sector'].values

    intra_sims, inter_sims = [], []
    for i in range(len(embeddings)):
        for j in range(i + 1, len(embeddings)):
            sim = float(np.dot(embeddings[i], embeddings[j]))
            if sectors[i] == sectors[j]:
                intra_sims.append(sim)
            else:
                inter_sims.append(sim)

    avg_intra = np.mean(intra_sims) if intra_sims else 0
    avg_inter = np.mean(inter_sims) if inter_sims else 0
    return avg_intra, avg_inter, avg_intra - avg_inter

print("\U0001f4ca Embedding Quality by Serialization Strategy")
print("=" * 70)
print(f"{'Strategy':<20} {'Intra-Sector':>14} {'Inter-Sector':>14} {'Gap (\u2191 better)':>16}")
print("-" * 70)

for name, func in serializers.items():
    intra, inter, gap = compute_sector_coherence(df, func, embed_model)
    print(f"{name:<20} {intra:>14.4f} {inter:>14.4f} {gap:>16.4f}")

print("\n\U0001f4a1 Higher gap means the embedding model better distinguishes sectors.")

---
# Section 4: Building the Complete CSV RAG Pipeline

Now we bring everything together into a working pipeline. We'll use the **natural language** serialization strategy (which typically offers the best balance of quality and simplicity) and build a complete system with:

1. **Embedding** all row representations into dense vectors
2. **Indexing** with FAISS for fast nearest-neighbor search
3. **Retrieval** with cosine similarity scoring
4. **Generation** using our quantized Qwen3-14B model with retrieved context

This is the core of any RAG system — the retrieval quality determines the answer quality.

In [ ]:
import faiss

# Step 1: Serialize all rows using natural language strategy
documents = []
for idx, row in df.iterrows():
    documents.append({
        'text': serialize_natural_language(row),
        'metadata': {
            'row_index': idx,
            'company': row['company'],
            'sector': row['sector'],
            'revenue': row['revenue_millions'],
            'employees': row['employees']
        }
    })

print(f"Created {len(documents)} documents from {len(df)} rows")
print(f"\nSample document:")
print(documents[0]['text'][:200] + "...")

In [ ]:
# Step 2: Embed all documents and build FAISS index
texts = [doc['text'] for doc in documents]
embeddings = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

# Build FAISS index (using inner product on normalized vectors = cosine similarity)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))

print(f"\u2705 FAISS index built: {index.ntotal} vectors, {dimension} dimensions")
print(f"   Index type: Flat Inner Product (exact cosine similarity on L2-normalized vectors)")

In [ ]:
# Step 3: Retrieval function with similarity scores
def retrieve(query, top_k=3, show_scores=True):
    """Retrieve the most relevant documents for a natural language query."""
    query_embedding = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        doc = documents[idx]
        results.append({
            'rank': rank + 1,
            'score': float(score),
            'text': doc['text'],
            'metadata': doc['metadata']
        })

    if show_scores:
        print(f"\U0001f50d Query: \"{query}\"")
        print(f"   Retrieved {top_k} results:")
        for r in results:
            print(f"   #{r['rank']} [score={r['score']:.4f}] {r['metadata']['company']} ({r['metadata']['sector']})")

    return results

# Test retrieval
retrieve("Tell me about artificial intelligence companies")

In [ ]:
# Step 4: RAG query function \u2014 retrieve context, then generate
def rag_query(question, top_k=3, max_new_tokens=512):
    """Full RAG pipeline: retrieve relevant rows, then generate an answer."""
    results = retrieve(question, top_k=top_k, show_scores=True)

    # Build context from retrieved documents
    context_parts = []
    for r in results:
        context_parts.append(f"[Company #{r['rank']}]\n{r['text']}")
    context = "\n\n".join(context_parts)

    # Build the RAG prompt
    prompt = f"""You are an expert business analyst. Answer the user's question using ONLY the company data provided below. If the data doesn't contain enough information, say so.

=== RETRIEVED COMPANY DATA ===
{context}

=== QUESTION ===
{question}

=== ANSWER ===
Provide a clear, detailed answer based on the data above."""

    print(f"\n\U0001f4dd Generating answer (context: {len(context)} chars, {top_k} docs)...\n")
    answer = generate(prompt, max_new_tokens=max_new_tokens)
    print(f"\U0001f4ac Answer:\n{answer}")
    return answer

# Test the full pipeline
rag_query("Which companies have the highest revenue?")

In [ ]:
# Query 2: Semantic query about a specific domain
rag_query("Tell me about healthcare companies and what services they provide")

In [ ]:
# Query 3: Query requiring synthesis across multiple rows
rag_query("Which companies are working on AI and machine learning? Compare their sizes.")

In [ ]:
# Query 4: Geographic query
rag_query("What companies are based in Texas?")

## Understanding Retrieval Scores

Let's dig deeper into how cosine similarity scores distribute across our dataset. This helps us understand the retrieval quality and set appropriate thresholds.

In practice, you'd use this analysis to decide:
- **Score threshold**: Below what score should results be filtered out?
- **Top-k selection**: How many results are typically relevant?
- **Query reformulation**: Are some query types consistently harder for retrieval?

In [ ]:
# Analyze score distributions for different query types
test_queries = [
    ("Exact match", "TechCorp AI software company"),
    ("Sector query", "renewable energy and sustainability"),
    ("Numerical", "companies with more than 5000 employees"),
    ("Vague", "small startups doing interesting things"),
    ("Cross-domain", "companies using technology in traditional industries"),
]

print("\U0001f4ca Score Distribution Analysis")
print("=" * 80)

for query_type, query in test_queries:
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, len(documents))
    all_scores = scores[0]

    top3 = all_scores[:3]
    rest = all_scores[3:]
    gap = top3[-1] - rest[0] if len(rest) > 0 else 0

    print(f"\n\U0001f50d {query_type}: \"{query}\"")
    print(f"   Top-3 scores:  {', '.join(f'{s:.4f}' for s in top3)}")
    print(f"   Score range:   [{all_scores.min():.4f}, {all_scores.max():.4f}]")
    print(f"   Top-3 / #4 gap: {gap:.4f} {'\u2705 clear separation' if gap > 0.05 else '\u26a0\ufe0f  weak separation'}")
    top_companies = [documents[idx]['metadata']['company'] for idx in indices[0][:3]]
    print(f"   Top matches:   {', '.join(top_companies)}")

---
# Section 5: Advanced CSV RAG Techniques

The basic pipeline works well for straightforward semantic queries, but real-world tabular RAG faces harder challenges:

1. **Hybrid queries** that combine structured filters with semantic search ("Show me Technology companies similar to TechCorp")
2. **Column-specific queries** that target one aspect of the data ("Which company has the most employees?")
3. **Numerical reasoning** where retrieval alone isn't enough ("What's the total revenue of healthcare companies?")

These require techniques beyond simple vector similarity. Let's build each one.

In [ ]:
# Advanced Technique 1: Hybrid Filtering + Semantic Search
# First filter by structured criteria, then semantic search within the filtered set

def hybrid_retrieve(query, filters=None, top_k=3):
    """Combine structured filtering with semantic search.

    Args:
        query: Natural language query for semantic matching
        filters: Dict of column -> value filters (applied pre-retrieval)
        top_k: Number of results to return
    """
    # Step 1: Apply structured filters to get candidate rows
    if filters:
        mask = pd.Series([True] * len(df))
        for col, val in filters.items():
            if col in df.columns:
                if isinstance(val, (list, tuple)):
                    mask &= df[col].isin(val)
                elif isinstance(val, dict):  # range filter: {'min': x, 'max': y}
                    if 'min' in val:
                        mask &= df[col] >= val['min']
                    if 'max' in val:
                        mask &= df[col] <= val['max']
                else:
                    mask &= df[col] == val
        candidate_indices = df[mask].index.tolist()
        print(f"\U0001f527 Structured filter: {filters}")
        print(f"   Candidates after filter: {len(candidate_indices)} / {len(df)} rows")
    else:
        candidate_indices = list(range(len(df)))

    if not candidate_indices:
        print("\u26a0\ufe0f  No rows match the filter criteria.")
        return []

    # Step 2: Semantic search within candidates
    candidate_embeddings = embeddings[candidate_indices].astype(np.float32)

    # Build a temporary FAISS index for candidates only
    temp_index = faiss.IndexFlatIP(dimension)
    temp_index.add(candidate_embeddings)

    query_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    k = min(top_k, len(candidate_indices))
    scores, local_indices = temp_index.search(query_emb, k)

    results = []
    print(f"\n\U0001f50d Semantic search within filtered set:")
    for rank, (score, local_idx) in enumerate(zip(scores[0], local_indices[0])):
        global_idx = candidate_indices[local_idx]
        doc = documents[global_idx]
        results.append({'rank': rank + 1, 'score': float(score), 'text': doc['text'], 'metadata': doc['metadata']})
        print(f"   #{rank+1} [score={score:.4f}] {doc['metadata']['company']} ({doc['metadata']['sector']})")

    return results

# Example: Find technology companies similar to AI/ML
results = hybrid_retrieve(
    query="artificial intelligence and machine learning platforms",
    filters={'sector': 'Technology'},
    top_k=3
)

In [ ]:
# Example: Find large companies (>2000 employees) in innovative fields
results = hybrid_retrieve(
    query="innovative cutting-edge technology and research",
    filters={'employees': {'min': 2000}},
    top_k=5
)

In [ ]:
# Advanced Technique 2: Column-Specific Queries
# Detect which columns a query targets, then use appropriate retrieval

def column_aware_query(question, top_k=3, max_new_tokens=512):
    """Handle queries that target specific columns by augmenting context
    with relevant aggregated statistics.
    """
    # Detect numerical intent with simple keyword matching
    numerical_keywords = {
        'revenue': 'revenue_millions',
        'employee': 'employees',
        'founded': 'founded',
        'oldest': 'founded',
        'newest': 'founded',
        'biggest': 'revenue_millions',
        'smallest': 'revenue_millions',
        'largest': 'employees',
        'most': 'revenue_millions',
        'highest': 'revenue_millions',
        'lowest': 'revenue_millions',
        'average': 'revenue_millions',
        'total': 'revenue_millions',
    }

    q_lower = question.lower()
    detected_columns = set()
    for keyword, col in numerical_keywords.items():
        if keyword in q_lower:
            detected_columns.add(col)

    # Retrieve semantically relevant documents
    results = retrieve(question, top_k=top_k, show_scores=True)

    # Build context
    context_parts = []
    for r in results:
        context_parts.append(f"[Company #{r['rank']}]\n{r['text']}")

    # If numerical columns detected, add aggregated statistics
    if detected_columns:
        stats_lines = ["\n[AGGREGATE STATISTICS]"]
        for col in detected_columns:
            if col in df.columns and df[col].dtype in ['int64', 'float64']:
                stats_lines.append(f"Column '{col}': min={df[col].min()}, max={df[col].max()}, "
                                   f"mean={df[col].mean():.1f}, median={df[col].median()}")
                # Add top/bottom 3
                sorted_df = df.sort_values(col, ascending=False)
                top3 = sorted_df.head(3)[['company', col]].to_string(index=False)
                stats_lines.append(f"Top 3 by {col}:\n{top3}")
        context_parts.append('\n'.join(stats_lines))
        print(f"\n\U0001f4ca Detected numerical columns: {detected_columns}")

    context = "\n\n".join(context_parts)
    prompt = f"""You are an expert business analyst. Answer the question using the company data AND statistics below. Be precise with numbers.

=== DATA ===
{context}

=== QUESTION ===
{question}

=== ANSWER ==="""

    print(f"\n\U0001f4dd Generating answer...\n")
    answer = generate(prompt, max_new_tokens=max_new_tokens)
    print(f"\U0001f4ac Answer:\n{answer}")
    return answer

# Test: numerical query that benefits from aggregated stats
column_aware_query("What is the average revenue and which company has the highest revenue?")

In [ ]:
# Advanced Technique 3: Handling Aggregation Queries
# Some questions need computation, not just retrieval

def aggregation_query(question, max_new_tokens=512):
    """Handle questions that need pandas-level aggregation before LLM synthesis."""

    # Provide the LLM with the full DataFrame summary for aggregation questions
    summary_parts = [
        f"Dataset overview: {len(df)} companies across {df['sector'].nunique()} sectors.\n",
        "Full company listing:",
        df[['company', 'sector', 'revenue_millions', 'employees', 'founded', 'hq_city']].to_string(index=False),
        f"\nRevenue statistics: mean=${df['revenue_millions'].mean():.1f}M, "
        f"median=${df['revenue_millions'].median():.1f}M, total=${df['revenue_millions'].sum()}M",
        f"Employee statistics: mean={df['employees'].mean():.0f}, total={df['employees'].sum():,}",
        f"\nSector breakdown:",
        df.groupby('sector').agg(
            count=('company', 'count'),
            avg_revenue=('revenue_millions', 'mean'),
            total_employees=('employees', 'sum')
        ).to_string(),
    ]
    context = '\n'.join(summary_parts)

    prompt = f"""You are a data analyst. Answer the question using the complete dataset summary below. Show your reasoning with specific numbers.

=== COMPLETE DATASET ===
{context}

=== QUESTION ===
{question}

=== ANSWER ==="""

    print(f"\U0001f4dd Aggregation query \u2014 providing full dataset context ({len(context)} chars)")
    answer = generate(prompt, max_new_tokens=max_new_tokens)
    print(f"\n\U0001f4ac Answer:\n{answer}")
    return answer

# Test: aggregation question
aggregation_query("What is the total revenue by sector? Which sector is the most profitable?")

---
# Section 6: Evaluation & Synthesis

## Quantitative Comparison of Serialization Strategies

Let's run a systematic evaluation. We'll test each serialization strategy against a set of diverse queries and measure retrieval quality using **Mean Reciprocal Rank (MRR)** — a standard IR metric that rewards strategies where the correct answer appears higher in the result list.

For each test query, we define the expected "ground truth" company (the one that should be ranked #1). MRR is 1/rank — so a strategy that always gets it right on the first try has MRR = 1.0.

In [ ]:
# Define evaluation queries with expected ground truth
eval_queries = [
    ("AI and machine learning software", "TechCorp"),
    ("solar energy and battery storage", "GreenEnergy"),
    ("telemedicine and digital health", "HealthPlus"),
    ("algorithmic trading and finance", "FinanceHub"),
    ("personalized education technology", "EduLearn"),
    ("self-driving vehicles", "AutoDrive"),
    ("cloud computing and DevOps", "CloudNine"),
    ("gene editing and cancer therapy", "BioGenesis"),
    ("retail stores and supply chain", "RetailMax"),
    ("satellite launch and space debris", "SpaceVentures"),
    ("precision farming and crop optimization", "AgriTech"),
    ("streaming media and content recommendation", "MediaFlow"),
    ("zero-trust cybersecurity", "CyberShield"),
    ("quantum computing research", "QuantumLabs"),
    ("farm-to-table food distribution", "FoodChain"),
]

def evaluate_strategy(serializer, queries, emb_model):
    """Compute MRR for a serialization strategy."""
    # Embed all rows with this strategy
    texts = [serializer(row) for _, row in df.iterrows()]
    embs = emb_model.encode(texts, normalize_embeddings=True).astype(np.float32)

    # Build index
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)

    reciprocal_ranks = []
    for query, expected_company in queries:
        q_emb = emb_model.encode([query], normalize_embeddings=True).astype(np.float32)
        _, result_indices = idx.search(q_emb, len(df))

        # Find rank of expected company
        expected_idx = df[df['company'] == expected_company].index[0]
        rank = list(result_indices[0]).index(expected_idx) + 1
        reciprocal_ranks.append(1.0 / rank)

    return np.mean(reciprocal_ranks)

print("\U0001f4ca Serialization Strategy Evaluation (MRR @ 15 queries)")
print("=" * 55)
for name, func in serializers.items():
    mrr = evaluate_strategy(func, eval_queries, embed_model)
    bar = "\u2588" * int(mrr * 30)
    print(f"  {name:<20} MRR = {mrr:.4f}  {bar}")

print("\n\U0001f4a1 MRR = 1.0 means the correct company is always ranked #1.")

## When CSV RAG Works vs. When SQL Is Better

Having built and evaluated our CSV RAG pipeline, we can now make informed judgments about when each approach excels:

**CSV RAG excels at:**
- Semantic similarity queries: "Find companies similar to TechCorp" — this requires understanding *what* TechCorp does, not just matching exact values
- Free-text field search: When your CSV has description/notes columns, embedding-based search captures meaning that keyword search would miss
- Cross-column reasoning: "Which tech companies in Texas focus on innovation?" — this blends category, geography, and description
- Fuzzy matching: "renewable energy" matching "solar panel technology" requires semantic understanding

**SQL is better for:**
- Exact aggregations: `SELECT AVG(revenue) FROM companies WHERE sector='Technology'` is precise and fast
- Range queries: `WHERE revenue > 500 AND employees < 3000` — structured filters are SQL's strength
- Joins across tables: SQL handles relational data naturally; RAG works on flat tables
- Sorted/ranked results: `ORDER BY revenue DESC LIMIT 5` is trivial in SQL

**The hybrid approach is strongest** — use structured SQL-like filters to narrow candidates, then apply semantic search. This is what our `hybrid_retrieve` function implements: it gets the precision of structured filtering with the flexibility of semantic understanding.

## Production Considerations for Tabular RAG

Deploying tabular RAG in production introduces challenges beyond what we've covered in this notebook. Here are the key considerations:

**Scalability:** Our FAISS flat index does exact search in O(n) time. For tables with millions of rows, switch to approximate methods like `IndexIVFFlat` or `IndexHNSWFlat`. The tradeoff is slightly lower recall for dramatically faster search.

**Schema Evolution:** Real CSV data changes over time — columns get added, renamed, or removed. Your serialization functions need to be robust to schema changes. Consider versioning your serializers and re-embedding when the schema changes significantly.

**Numerical Precision:** Embedding models are notoriously bad at understanding numbers. The query "companies with revenue over 500 million" may not reliably retrieve the right results through pure vector search. The hybrid approach (filter numerically, then search semantically) is essential for production systems.

**Update Patterns:** When new rows are added, you can incrementally add to the FAISS index without re-embedding everything. But if you change serialization strategy, you must re-embed the entire dataset. Design your pipeline with this in mind.

**Evaluation at Scale:** Our MRR evaluation used hand-crafted queries. In production, build automated evaluation sets and track retrieval quality over time as data changes.

## Limitations and Failure Modes

Understanding where CSV RAG fails is as important as understanding where it works:

1. **Numerical reasoning**: "What's the ratio of TechCorp's revenue to GreenEnergy's?" — retrieval finds both companies, but the LLM must do the math. Results are unreliable for complex arithmetic.

2. **Negation queries**: "Which companies are NOT in the Technology sector?" — semantic search looks for similarity, so it actually retrieves Technology companies (the most semantically similar to the query).

3. **Complete enumeration**: "List ALL companies founded after 2015" — retrieval returns top-k, which may miss some. You need structured filtering for exhaustive queries.

4. **Cross-row comparison**: "Is TechCorp older than CloudNine?" — requires retrieving both specific rows and comparing a field. Simple top-k retrieval may not surface both.

5. **Ambiguous queries**: "Tell me about the biggest company" — biggest by revenue? Employees? Market cap? The embedding can't resolve this ambiguity.

These limitations aren't bugs — they're fundamental properties of semantic search over structured data. The best production systems use **RAG as one tool in a toolkit**, combining it with structured queries, explicit filters, and computational steps as needed.

In [ ]:
# Demonstration: A failure mode \u2014 negation queries
print("\U0001f534 Failure Mode: Negation Queries")
print("Query: 'Companies that are NOT in the technology sector'")
print()
results = retrieve("Companies that are NOT in the technology sector", top_k=5)
print()
tech_count = sum(1 for r in results if r['metadata']['sector'] == 'Technology')
print(f"\u26a0\ufe0f  {tech_count}/{len(results)} results are Technology companies!")
print("This happens because 'NOT technology' is still semantically close to 'technology'.")

In [ ]:
# Demonstration: Hybrid approach overcomes the negation limitation
print("\U0001f7e2 Solution: Hybrid filter + semantic search")
print("Filter: sector NOT in ['Technology'], then search semantically")
print()

non_tech_sectors = [s for s in df['sector'].unique() if s != 'Technology']
results = hybrid_retrieve(
    query="innovative companies with strong growth potential",
    filters={'sector': non_tech_sectors},
    top_k=5
)
print("\n\u2705 All results are correctly non-Technology companies!")

---
## Summary

In this notebook we built a **complete RAG pipeline for structured CSV data** from first principles:

| Component | What We Built |
|-----------|---------------|
| **Serialization** | 5 strategies for converting rows to text (naive \u2192 schema-enriched) |
| **Chunking** | Row-level, group-level, and schema-enriched chunking |
| **Embedding** | BGE-base-en-v1.5 with FAISS inner-product index |
| **Retrieval** | Cosine similarity search with score analysis |
| **Generation** | Qwen3-14B (4-bit NF4) with retrieved context injection |
| **Advanced** | Hybrid filtering, column-aware queries, aggregation handling |
| **Evaluation** | MRR-based strategy comparison, failure mode analysis |

**Key takeaway:** The serialization strategy is the most important design choice in tabular RAG. Natural language serialization consistently outperforms naive approaches because it aligns the data representation with what embedding models were trained on.